# Qwen2.5-VL-7B VQA Fine-tuning + Seed Ensemble (Colab Pro+)

**버그 수정 + 성능 개선 버전**

| 수정 항목 | 내용 |
|-----------|------|
| 🔴 Bug 1 | `total_mem` → `total_memory` 오타 수정 |
| 🔴 Bug 2 | 추론 시 input 토큰 슬라이싱 누락 수정 |
| 🔴 Bug 3 | `add_generation_prompt` 추론 시 True로 수정 |
| 🟡 개선 1 | 레이블 마스킹: assistant 답변 토큰에만 loss 적용 |
| 🟡 개선 2 | `min_pixels` 분리 (작은 이미지 강제 업스케일 방지) |
| 🟡 개선 3 | 추론 배치화 (1개씩 → DataLoader 배치, ~10배 빠름) |
| 🟡 개선 4 | GradScaler 추가 (학습 안정성) |

## 1. 환경 설치

In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" datasets pandas==2.2.2 pillow==11.1.0
!pip -q install qwen-vl-utils

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 49.0 MB/s eta 0:00:00


In [2]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
# 🔴 Bug 1 수정: total_mem → total_memory
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
print("VRAM:", vram_gb, "GB")

if vram_gb < 50:
    print("\n⚠️  A100 40GB 감지 → 설정 섹션에서 보수적 값 사용 권장")
    print("   Runtime → Change runtime type → High RAM 슬라이더 ON 추천")
else:
    print("\n✅ A100 80GB — 현재 설정으로 안정적")

Torch: 2.10.0+cu128
CUDA: 12.8
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB

✅ A100 80GB — 현재 설정으로 안정적


## 2. Google Drive 마운트 & 데이터 준비

In [5]:
from google.colab import drive
drive.mount("/content/drive")

import os
PROJECT_DIR = "/content/drive/My Drive/vqa_project"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"프로젝트 폴더: {PROJECT_DIR}")

Mounted at /content/drive
프로젝트 폴더: /content/drive/My Drive/vqa_project


In [7]:
!unzip -q "/content/drive/My Drive/260401_15_2_ai_데이터배포용_기본_데이터ver.zip" -d "/content/"

## 3. 설정 & 라이브러리

**A100 VRAM별 권장 설정:**
| GPU | IMAGE_SIZE | BATCH_SIZE | GRAD_ACCUM | LORA_R |
|-----|-----------|------------|------------|--------|
| A100 80GB | 768 | 2 | 8 | 16 |
| A100 40GB | 512 | 1 | 16 | 8 |

In [10]:
import os, re, math, random, gc, json, shutil, time
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ===== 핵심 설정 =====
# ⚠️ A100 40GB면 IMAGE_SIZE=512, BATCH_SIZE=1, GRAD_ACCUM=16, LORA_R=8
MODEL_ID   = "Qwen/Qwen2.5-VL-7B-Instruct"
IMAGE_SIZE = 768       # A100 80GB 기준
EPOCHS     = 3
BATCH_SIZE = 2         # 80GB: 2, 40GB: 1
GRAD_ACCUM = 8         # effective batch = 16
LR         = 2e-5      # VLM fine-tuning 표준
LORA_R     = 16
LORA_ALPHA = 32
SEEDS      = [42, 123, 2024]

# Google Drive 경로
PROJECT_DIR = "/content/drive/My Drive/vqa_project"

# 데이터 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# 정답 정규화
train_df['answer'] = train_df['answer'].astype(str).str.strip().str.lower()
train_df = train_df[train_df['answer'].isin(['a','b','c','d'])].reset_index(drop=True)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"정답 분포:\n{train_df['answer'].value_counts().sort_index()}")

Device: cuda
Train: 5073, Test: 5074
정답 분포:
answer
a    1263
b    1265
c    1311
d    1234
Name: count, dtype: int64


## 4. 세션 복구용 유틸리티

In [11]:
def save_json(path, data):
    for attempt in range(3):
        try:
            with open(path, "w") as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            return
        except OSError:
            time.sleep(2)
    print(f"⚠️ JSON 저장 실패: {path}")

def load_json(path, default=None):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default if default is not None else {}

def get_progress():
    return load_json(f"{PROJECT_DIR}/progress.json",
                     default={"completed_seeds": [], "all_done": False})

def save_progress(progress):
    save_json(f"{PROJECT_DIR}/progress.json", progress)

def get_seed_state(seed):
    seed_dir = f"{PROJECT_DIR}/seed{seed}"
    return load_json(f"{seed_dir}/train_state.json",
                     default={"completed_epoch": 0,
                              "best_val_loss": float("inf"),
                              "inference_done": False})

def save_seed_state(seed, state):
    seed_dir = f"{PROJECT_DIR}/seed{seed}"
    os.makedirs(seed_dir, exist_ok=True)
    save_json(f"{seed_dir}/train_state.json", state)

print("✅ 유틸리티 로드 완료")
prog = get_progress()
print(f"완료된 시드: {prog.get('completed_seeds', [])}")
print(f"전체 완료: {prog.get('all_done', False)}")

✅ 유틸리티 로드 완료
완료된 시드: [42, 123, 2024]
전체 완료: True


## 5. 프롬프트 & 데이터셋

In [12]:
SYSTEM_INSTRUCT = (
    "You are a precise visual question answering assistant. "
    "Look at the image carefully and answer the multiple choice question. "
    "Output exactly one lowercase letter: a, b, c, or d. Nothing else."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Answer with a single letter only (a/b/c/d):"
    )


class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        user_text = build_mc_prompt(
            str(row["question"]), str(row["a"]),
            str(row["b"]), str(row["c"]), str(row["d"])
        )
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [{"type": "image", "image": img},
                                            {"type": "text",  "text": user_text}]},
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})
        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                # 🔴 Bug 3 수정: 추론 시 add_generation_prompt=True
                add_generation_prompt=not self.train,
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text=texts, images=images,
            padding=True, return_tensors="pt"
        )

        if self.train:
            # 🟡 개선 1: assistant 답변 토큰에만 loss 적용
            labels = enc["input_ids"].clone()
            for b_idx in range(labels.shape[0]):
                seq = labels[b_idx]
                eos_positions = (seq == self.processor.tokenizer.eos_token_id).nonzero()
                if len(eos_positions) >= 2:
                    mask_until = eos_positions[-2].item() + 1
                else:
                    mask_until = max(0, seq.shape[0] - 3)
                labels[b_idx, :mask_until] = -100
            enc["labels"] = labels

        return enc


def extract_choice(text: str) -> str:
    text = text.strip().lower()
    # 첫 글자가 바로 정답인 경우 (가장 흔함)
    if text and text[0] in 'abcd':
        return text[0]
    # 줄별 검색
    for line in text.splitlines():
        line = line.strip()
        if line in ['a', 'b', 'c', 'd']:
            return line
    # 토큰별 검색
    for tok in re.split(r'[\s,.:;!?]', text):
        if tok in ['a', 'b', 'c', 'd']:
            return tok
    # fallback
    return 'a'


def majority_vote(all_preds):
    n = len(all_preds[0])
    return [Counter(p[i] for p in all_preds).most_common(1)[0][0] for i in range(n)]


print("✅ 데이터셋 & 유틸 정의 완료")

✅ 데이터셋 & 유틸 정의 완료


## 6. 모델 로드 & 저장 함수

In [13]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_fresh_model_and_processor():
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        # 🟡 개선 2: min_pixels 분리 (작은 이미지 강제 업스케일 방지)
        min_pixels=256 * 256,
        max_pixels=IMAGE_SIZE * IMAGE_SIZE,
        trust_remote_code=True,
    )
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    base_model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    return model, processor


def load_model_from_checkpoint(ckpt_dir):
    processor = AutoProcessor.from_pretrained(
        ckpt_dir,
        min_pixels=256 * 256,
        max_pixels=IMAGE_SIZE * IMAGE_SIZE,
        trust_remote_code=True,
    )
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    base_model.gradient_checkpointing_enable()
    model = PeftModel.from_pretrained(base_model, ckpt_dir, is_trainable=True)
    model.print_trainable_parameters()
    return model, processor


def save_checkpoint_to_drive(model, processor, seed, epoch, is_best=False):
    seed_dir = f"{PROJECT_DIR}/seed{seed}"
    save_path = f"{seed_dir}/best_model" if is_best else f"{seed_dir}/checkpoint_epoch{epoch}"
    local_tmp = f"/content/tmp_ckpt_seed{seed}"

    # 로컬 저장 후 Drive로 복사 (Drive 직접 쓰기 불안정 방지)
    os.makedirs(local_tmp, exist_ok=True)
    model.save_pretrained(local_tmp)
    processor.save_pretrained(local_tmp)

    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    shutil.copytree(local_tmp, save_path)
    shutil.rmtree(local_tmp)

    label = "BEST" if is_best else f"epoch{epoch}"
    print(f"  💾 [{label}] Drive 저장 완료 → {save_path}")


print("✅ 모델 함수 정의 완료")

✅ 모델 함수 정의 완료


## 7. 시드별 학습 + 추론 (자동 재개)

In [14]:
def run_batch_inference(model, processor, df, batch_size=8):
    """
    🟡 개선 3: 배치 추론 (기존 1개씩 루프 대비 ~10배 빠름)
    """
    inf_ds = VQAMCDataset(df, processor, train=False)
    inf_loader = DataLoader(
        inf_ds, batch_size=batch_size, shuffle=False,
        collate_fn=DataCollator(processor, train=False),
        num_workers=0,
    )
    preds = []
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for batch in tqdm(inf_loader, desc="배치 추론 중"):
            inputs = {k: v.to(device) for k, v in batch.items()}
            out_ids = model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                eos_token_id=processor.tokenizer.eos_token_id,
            )
            # 🔴 Bug 2 수정: input 길이만큼 슬라이싱해서 생성 토큰만 디코딩
            input_len = inputs["input_ids"].shape[1]
            generated = out_ids[:, input_len:]
            decoded = processor.batch_decode(generated, skip_special_tokens=True)
            preds.extend([extract_choice(t) for t in decoded])
    return preds


def train_one_seed(seed, train_df, test_df):
    seed_dir = f"{PROJECT_DIR}/seed{seed}"
    os.makedirs(seed_dir, exist_ok=True)
    state = get_seed_state(seed)

    # 이미 추론까지 완료된 시드면 결과만 로드
    preds_path = f"{seed_dir}/preds_seed{seed}.csv"
    if state.get("inference_done") and os.path.exists(preds_path):
        print(f"\n✅ [Seed {seed}] 이미 완료 — 저장된 결과 로드")
        return pd.read_csv(preds_path)["answer"].tolist()

    print(f"\n{'='*60}")
    print(f"  SEED = {seed} | 이어서 학습 (completed_epoch={state['completed_epoch']})")
    print(f"{'='*60}")

    set_seed(seed)

    # 모델 로드: 체크포인트 있으면 거기서, 없으면 새로
    last_epoch = state["completed_epoch"]
    ckpt_path  = f"{seed_dir}/checkpoint_epoch{last_epoch}"

    if last_epoch > 0 and os.path.exists(ckpt_path):
        print(f"  🔄 체크포인트에서 복원: epoch {last_epoch}")
        model, processor = load_model_from_checkpoint(ckpt_path)
    else:
        print("  🆕 새 모델로 시작")
        model, processor = load_fresh_model_and_processor()

    # 시드 기반 셔플 → 재현 보장
    shuffled  = train_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    split     = int(len(shuffled) * 0.9)
    train_sub = shuffled.iloc[:split]
    valid_sub = shuffled.iloc[split:]
    print(f"  Train: {len(train_sub)}, Valid: {len(valid_sub)}")

    train_loader = DataLoader(
        VQAMCDataset(train_sub, processor, train=True),
        batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=DataCollator(processor, True), num_workers=0,
    )
    valid_loader = DataLoader(
        VQAMCDataset(valid_sub, processor, train=True),
        batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=DataCollator(processor, True), num_workers=0,
    )

    optimizer  = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=0.01, betas=(0.9, 0.95)
    )
    total_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.05),
        num_training_steps=total_steps,
    )
    # 🟡 개선 4: GradScaler 추가
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    # 이미 완료된 에폭만큼 스케줄러 앞으로 감기
    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    for _ in range(last_epoch * steps_per_epoch):
        scheduler.step()

    best_val_loss = state.get("best_val_loss", float("inf"))

    # ===== 학습 루프 =====
    for epoch in range(last_epoch + 1, EPOCHS + 1):
        model.train()
        running = 0.0
        optimizer.zero_grad(set_to_none=True)
        progress = tqdm(train_loader, desc=f"[Seed {seed}] Epoch {epoch}/{EPOCHS} train")

        for step, batch in enumerate(progress, 1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                loss = model(**batch).loss / GRAD_ACCUM

            scaler.scale(loss).backward()
            running += loss.item()

            if step % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                progress.set_postfix({
                    "loss": f"{running:.4f}",
                    "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                })
                running = 0.0

        # Validation
        model.eval()
        val_loss, val_steps = 0.0, 0
        with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
            for vb in tqdm(valid_loader, desc=f"[Seed {seed}] Epoch {epoch} valid"):
                vb = {k: v.to(device) for k, v in vb.items()}
                val_loss += model(**vb).loss.item()
                val_steps += 1

        avg_val = val_loss / max(val_steps, 1)
        print(f"  [Epoch {epoch}] val_loss: {avg_val:.4f}")

        # 에폭 체크포인트 Drive 저장
        save_checkpoint_to_drive(model, processor, seed, epoch)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            save_checkpoint_to_drive(model, processor, seed, epoch, is_best=True)

        state["completed_epoch"] = epoch
        state["best_val_loss"]   = best_val_loss
        save_seed_state(seed, state)
        print(f"  📝 상태 저장: epoch={epoch}, best_val={best_val_loss:.4f}")

    # ===== 추론: best 모델로 배치 추론 =====
    best_path = f"{seed_dir}/best_model"
    if os.path.exists(best_path):
        print(f"\n  🏆 Best 모델로 추론: {best_path}")
        del model
        gc.collect(); torch.cuda.empty_cache()
        model, processor = load_model_from_checkpoint(best_path)

    # 🟡 개선 3: 배치 추론
    preds = run_batch_inference(model, processor, test_df, batch_size=8)

    # 추론 결과 Drive 저장
    pd.DataFrame({"id": test_df["id"], "answer": preds}).to_csv(preds_path, index=False)
    print(f"  💾 추론 결과 저장: {preds_path}")

    state["inference_done"] = True
    save_seed_state(seed, state)

    del model, processor
    gc.collect(); torch.cuda.empty_cache()

    return preds


print("✅ 학습 함수 정의 완료")

✅ 학습 함수 정의 완료


## 새 7번셀 (오전 10:33)**굵은 텍스트**

In [18]:
def run_batch_inference(model, processor, df, batch_size=8):
    """
    ✅ 패딩 방향 오류가 수정된 배치 추론 함수
    """
    # 🟢 핵심: 배치 추론 시 패딩을 왼쪽에 붙여야 모델이 오른쪽 끝에서 정답을 생성함
    processor.tokenizer.padding_side = "left"

    inf_ds = VQAMCDataset(df, processor, train=False)
    inf_loader = DataLoader(
        inf_ds, batch_size=batch_size, shuffle=False,
        collate_fn=DataCollator(processor, train=False),
        num_workers=0,
    )

    preds = []
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for batch in tqdm(inf_loader, desc="배치 추론 중"):
            inputs = {k: v.to(device) for k, v in batch.items()}
            out_ids = model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                eos_token_id=processor.tokenizer.eos_token_id,
            )
            # 입력 토큰 길이를 제외한 생성된 토큰만 슬라이싱
            input_len = inputs["input_ids"].shape[1]
            generated = out_ids[:, input_len:]
            decoded = processor.batch_decode(generated, skip_special_tokens=True)

            # 정답 추출 및 리스트 추가
            preds.extend([extract_choice(t) for t in decoded])

    return preds




In [19]:
all_seed_preds = []

for seed in SEEDS:
    seed_dir = f"{PROJECT_DIR}/seed{seed}"
    best_path = f"{seed_dir}/best_model"
    preds_path = f"{seed_dir}/preds_seed{seed}_fixed.csv" # 수정된 결과임을 표시

    print(f"\n{'='*60}")
    print(f"  SEED = {seed} | 체크포인트 로드 및 수정된 추론 시작")
    print(f"{'='*60}")

    if os.path.exists(best_path):
        # 1. 모델 및 프로세서 로드
        model, processor = load_model_from_checkpoint(best_path)

        # 2. 수정된 추론 함수 실행
        preds = run_batch_inference(model, processor, test_df, batch_size=8)

        # 3. 결과 저장
        pd.DataFrame({"id": test_df["id"], "answer": preds}).to_csv(preds_path, index=False)
        all_seed_preds.append(preds)
        print(f"  💾 Seed {seed} 추론 완료 및 저장: {preds_path}")

        # 메모리 해제 (VRAM 관리)
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()
    else:
        print(f"  ❌ Seed {seed}의 Best 모델을 찾을 수 없습니다. 경로를 확인하세요: {best_path}")

# ===== 앙상블 (Majority Vote) =====
if len(all_seed_preds) == len(SEEDS):
    final_preds = majority_vote(all_seed_preds)

    n = len(final_preds)
    agree = sum(1 for i in range(n) if all(p[i] == all_seed_preds[0][i] for p in all_seed_preds))

    print(f"\n3개 시드 완전 일치율: {agree}/{n} ({100*agree/n:.1f}%)")
    print(f"최종 예측 분포: {Counter(final_preds)}")

    submission = pd.DataFrame({"id": test_df["id"], "answer": final_preds})
    ensemble_path = f"{PROJECT_DIR}/submission_ensemble_fixed.csv"
    submission.to_csv(ensemble_path, index=False)

    print(f"\n🎉 수정된 최종 앙상블 파일 생성 완료: {ensemble_path}")
else:
    print("\n⚠️ 모든 시드의 추론 결과가 모이지 않아 앙상블을 진행할 수 없습니다.")


  SEED = 42 | 체크포인트 로드 및 수정된 추론 시작


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

trainable params: 47,589,376 || all params: 8,339,756,032 || trainable%: 0.5706


배치 추론 중:   0%|          | 0/635 [00:00<?, ?it/s]

  💾 Seed 2024 추론 완료 및 저장: /content/drive/My Drive/vqa_project/seed2024/preds_seed2024_fixed.csv

3개 시드 완전 일치율: 4858/5074 (95.7%)
최종 예측 분포: Counter({'c': 1345, 'd': 1332, 'b': 1330, 'a': 1067})

🎉 수정된 최종 앙상블 파일 생성 완료: /content/drive/My Drive/vqa_project/submission_ensemble_fixed.csv


## 8. 전체 실행 (자동 재개)

**세션이 끊기면?** → 런타임 재시작 후 셀 1~8을 순서대로 다시 실행하세요.  
이미 완료된 시드/에폭은 자동으로 스킵되고, 끊긴 지점부터 이어갑니다.

In [22]:
progress = get_progress()

if progress.get("all_done"):
    print("🎉 모든 시드 완료!")
    print(f"최종 파일: {PROJECT_DIR}/submission_ensemble.csv")
else:
    all_seed_preds = []

    for seed in SEEDS:
        preds_path = f"{PROJECT_DIR}/seed{seed}/preds_seed{seed}.csv"

        if seed in progress.get("completed_seeds", []) and os.path.exists(preds_path):
            print(f"✅ [Seed {seed}] 이전에 완료됨 — 결과 로드")
            all_seed_preds.append(pd.read_csv(preds_path)["answer"].tolist())
        else:
            preds = train_one_seed(seed, train_df, test_df)
            all_seed_preds.append(preds)
            progress.setdefault("completed_seeds", []).append(seed)
            save_progress(progress)
            print(f"✅ [Seed {seed}] 완료 & 저장됨")

    # ===== 앙상블 =====
    final_preds = majority_vote(all_seed_preds)

    n = len(final_preds)
    agree = sum(
        1 for i in range(n)
        if all(p[i] == all_seed_preds[0][i] for p in all_seed_preds)
    )
    print(f"\n3개 시드 완전 일치율: {agree}/{n} ({100*agree/n:.1f}%)")
    print(f"최종 예측 분포: {Counter(final_preds)}")

    submission = pd.DataFrame({"id": test_df["id"], "answer": final_preds})
    ensemble_path = f"{PROJECT_DIR}/submission_ensemble.csv"
    submission.to_csv(ensemble_path, index=False)
    submission.to_csv("/content/submission_ensemble.csv", index=False)

    progress["all_done"] = True
    save_progress(progress)

    print(f"\n🎉 최종 앙상블 제출 파일: {ensemble_path}")
    submission.head(10)

🎉 모든 시드 완료!
최종 파일: /content/drive/My Drive/vqa_project/submission_ensemble.csv


## 9. 진행 상태 확인 & 리셋

In [21]:
print("=" * 50)
print("📊 전체 진행 상태")
print("=" * 50)

prog = get_progress()
print(f"완료된 시드: {prog.get('completed_seeds', [])}")
print(f"전체 완료: {prog.get('all_done', False)}")
print()

for seed in SEEDS:
    state       = get_seed_state(seed)
    preds_exists = os.path.exists(f"{PROJECT_DIR}/seed{seed}/preds_seed{seed}.csv")
    best_exists  = os.path.exists(f"{PROJECT_DIR}/seed{seed}/best_model")
    print(f"[Seed {seed}]")
    print(f"  완료 에폭  : {state.get('completed_epoch', 0)} / {EPOCHS}")
    print(f"  Best val   : {state.get('best_val_loss', 'N/A')}")
    print(f"  Best 모델  : {'✅' if best_exists else '❌'}")
    print(f"  추론 완료  : {'✅' if state.get('inference_done') else '❌'}")
    print(f"  결과 CSV   : {'✅' if preds_exists else '❌'}")
    print()

📊 전체 진행 상태
완료된 시드: [42, 123, 2024]
전체 완료: True

[Seed 42]
  완료 에폭  : 3 / 3
  Best val   : 0.022736679117608977
  Best 모델  : ✅
  추론 완료  : ✅
  결과 CSV   : ✅

[Seed 123]
  완료 에폭  : 3 / 3
  Best val   : 0.02076898887859654
  Best 모델  : ✅
  추론 완료  : ✅
  결과 CSV   : ✅

[Seed 2024]
  완료 에폭  : 3 / 3
  Best val   : 0.01739180603177252
  Best 모델  : ✅
  추론 완료  : ✅
  결과 CSV   : ✅



In [17]:
# ⚠️ 특정 시드 리셋 (필요할 때만 주석 해제)

# RESET_SEED = 42
# seed_dir = f"{PROJECT_DIR}/seed{RESET_SEED}"
# if os.path.exists(seed_dir):
#     shutil.rmtree(seed_dir)
#     print(f"🗑️ Seed {RESET_SEED} 리셋 완료")
#     prog = get_progress()
#     if RESET_SEED in prog.get("completed_seeds", []):
#         prog["completed_seeds"].remove(RESET_SEED)
#         prog["all_done"] = False
#         save_progress(prog)
#         print("진행 상태 업데이트 완료")

## 10. OOM 발생 시 대처

OOM이 발생하면 **셀 3**의 설정을 아래처럼 변경 후 해당 시드를 리셋하세요.

```python
# 1순위: 이미지 사이즈 축소
IMAGE_SIZE = 512

# 2순위: 배치 & LoRA 축소
BATCH_SIZE = 1
GRAD_ACCUM = 16
LORA_R = 8; LORA_ALPHA = 16

# 3순위 (최후 수단): 8bit 양자화
# load_fresh_model_and_processor() 내부에서:
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
# from_pretrained(..., quantization_config=bnb_config)
```

또는 **Runtime → Change runtime type → High RAM ON** 으로 A100 80GB를 받으세요.